# Show a GNoME structure

Change `FORMULA` and Run All.

Cell 4 draws the structure **inside the notebook** as a static image — that one always works.
Cell 5 writes a **self-contained interactive HTML file** you open in a browser; 3Dmol.js is
embedded in it, so it needs no internet and no Jupyter.

(JupyterLab blocks the `<script>` tags that in-notebook 3D viewers rely on, which is why
nglview and inline py3Dmol both come up blank here. These two routes sidestep it.)

In [ ]:
FORMULA = "Ba2Sr6Sb4H2O"

In [ ]:
import os, zipfile, warnings
import pandas as pd
warnings.filterwarnings("ignore")
from pymatgen.core import Composition, Structure

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

# 168 MB - loaded once per session. dtype=str because 34,046 GNoME ids
# begin with a leading zero and would lose it to number parsing.
if "SUMMARY" not in globals():
    SUMMARY = pd.read_csv(os.path.join(ROOT, "gnome_data", "stable_materials_summary.csv"),
                          dtype={"MaterialId": str})

target = Composition(FORMULA).reduced_formula
hits = SUMMARY[SUMMARY["Reduced Formula"].map(
    lambda f: pd.notna(f) and Composition(str(f)).reduced_formula == target)]

if hits.empty:
    raise SystemExit(f"{FORMULA} is not in GNoME")

row = hits.iloc[0]
z = zipfile.ZipFile(os.path.join(ROOT, "gnome_data", "by_composition.zip"))
cif_text = z.read(f"by_composition/{row['Composition']}.CIF").decode()
st = Structure.from_str(cif_text, fmt="cif")

print(f"{row['Reduced Formula']}   {row['MaterialId']}   {row['Space Group']}   {len(st)} sites")
if len(hits) > 1:
    print(f"({len(hits)} matches - showing the first; use row = hits.iloc[n] for the others)")

In [ ]:
# ---- static view: a PNG, so nothing can block it -------------------------
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  registers the 3d projection

fig = plt.figure(figsize=(7.2, 6.2))
ax = fig.add_subplot(111, projection="3d")

species = sorted({s.specie.symbol for s in st})
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
colour = {s: palette[i % len(palette)] for i, s in enumerate(species)}

for sym in species:
    pts = [s.coords for s in st if s.specie.symbol == sym]
    xs, ys, zs = zip(*pts)
    ax.scatter(xs, ys, zs, s=190, color=colour[sym], edgecolor="white",
               linewidth=0.8, label=sym, depthshade=True)

# unit cell edges, so the atoms sit in a box rather than in space
m = st.lattice.matrix
corner = {(i, j, k): i * m[0] + j * m[1] + k * m[2]
          for i in (0, 1) for j in (0, 1) for k in (0, 1)}
for (i, j, k), p0 in corner.items():
    for di, dj, dk in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]:
        n = (i + di, j + dj, k + dk)
        if n in corner:
            ax.plot(*zip(p0, corner[n]), color="#9aa0a6", lw=0.8)

ax.set_title(f"{st.composition.reduced_formula}   {row['MaterialId']}   "
             f"{row['Space Group']}\n{len(st)} sites", fontsize=11)
ax.set_xlabel("x (A)"); ax.set_ylabel("y (A)"); ax.set_zlabel("z (A)")
ax.legend(loc="upper left", fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import json
# ---- interactive view: one self-contained .html you open in a browser -----
# 3Dmol.js is INLINED into the file (vendored at notebooks/vendor/), so the
# page works offline and does not depend on a CDN or on JupyterLab's willingness
# to run scripts in an output cell - which is what defeated every in-notebook
# attempt here.
js = open(os.path.join(ROOT, "notebooks", "vendor", "3Dmol-min.js"), encoding="utf-8").read()

html = f"""<!doctype html>
<meta charset="utf-8">
<title>{target} - {row['MaterialId']}</title>
<style>
  body {{ margin:0; font:14px/1.5 system-ui,sans-serif; background:#fff; color:#15181d; }}
  header {{ padding:14px 18px; border-bottom:1px solid #dce1e9; }}
  h1 {{ margin:0 0 3px; font-size:17px; }}
  .meta {{ color:#6b7484; font-size:13px; }}
  #v {{ width:100vw; height:calc(100vh - 68px); position:relative; }}
</style>
<header>
  <h1>{target}</h1>
  <div class="meta">{row['MaterialId']} &middot; {row['Space Group']} (#{row['Space Group Number']})
    &middot; {row['Crystal System']} &middot; {len(st)} sites &middot; {st.volume:.1f} A&sup3;</div>
</header>
<div id="v"></div>
<script>{js}</script>
<script>
  const cif = {json.dumps(cif_text)};
  const viewer = $3Dmol.createViewer(document.getElementById("v"), {{backgroundColor:"white"}});
  viewer.addModel(cif, "cif");
  viewer.setStyle({{}}, {{sphere:{{scale:0.34}}, stick:{{radius:0.13}}}});
  viewer.addUnitCell();
  viewer.zoomTo();
  viewer.render();
</script>
"""

out = os.path.join(ROOT, "notebooks", f"view_{target}.html")
with open(out, "w", encoding="utf-8") as fh:
    fh.write(html)

print(f"wrote {os.path.relpath(out, ROOT)}  ({len(html)/1e6:.1f} MB, fully self-contained)")
print("Open it: double-click in the Jupyter file browser, or")
print(f"  open '{out}'")